# HAVEN — Eval Analysis
**Notebook:** 02_eval_analysis.ipynb  
**Purpose:** Compare pipeline outputs against the golden set + run LLM-as-Judge on Agent 3 explanations  
**Judge model:** GPT-4.1 (OpenAI)  
**Run after:** 01_pipeline.ipynb

---
This notebook covers two evaluations:
1. **Match Accuracy** — did Agent 2 recommend the right helpers vs. golden set expected_top_3?
2. **Explanation Quality** — does GPT-4.1 judge Agent 3 explanations as specific, relevant, factually grounded, and empathetic?

## 1. Setup

In [1]:
import json
import os
import time
from pathlib import Path
from openai import OpenAI

# Paths
BASE_DIR = Path("../")
DATA_DIR = BASE_DIR / "data"
PIPELINE_DIR = Path("outputs/pipeline")
JUDGE_DIR = Path("outputs/judge")
JUDGE_DIR.mkdir(parents=True, exist_ok=True)

# OpenAI client for Judge
openai_client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

print("Setup complete.")

Setup complete.


## 2. Load All Data

In [2]:
# Load golden set
with open(DATA_DIR / "golden_set.json") as f:
    golden_set = json.load(f)

with open(DATA_DIR / "helpers.json") as f:
    helpers = json.load(f)

with open(DATA_DIR / "families.json") as f:
    families = json.load(f)

# Load pipeline outputs
with open(PIPELINE_DIR / "agent_1_outputs.json") as f:
    agent_1_outputs = json.load(f)

with open(PIPELINE_DIR / "agent_2_outputs.json") as f:
    agent_2_outputs = json.load(f)

with open(PIPELINE_DIR / "agent_3_outputs.json") as f:
    agent_3_outputs = json.load(f)

# Build lookup maps
golden_lookup = {g["family_id"]: g for g in golden_set}
a1_lookup = {r["family_id"]: r["care_profile"] for r in agent_1_outputs}
a2_lookup = {r["family_id"]: r["matching_result"] for r in agent_2_outputs}
a3_lookup = {r["family_id"]: r["recommendations"] for r in agent_3_outputs}
helper_lookup = {h["helper_id"]: h for h in helpers}
family_lookup = {f["family_id"]: f for f in families}

print(f"Golden set: {len(golden_set)} entries")
print(f"Agent 1 outputs: {len(agent_1_outputs)}")
print(f"Agent 2 outputs: {len(agent_2_outputs)}")
print(f"Agent 3 outputs: {len(agent_3_outputs)}")

Golden set: 17 entries
Agent 1 outputs: 17
Agent 2 outputs: 17
Agent 3 outputs: 17


## 3. Match Accuracy — Agent 2 vs. Golden Set
For each family, compare Agent 2's top_3 helper IDs against the golden set's expected_top_3.  
Metrics: exact rank match, top-3 set match (right helpers regardless of order), rank-1 match.

In [3]:
match_results = []

for family_id, a2_result in a2_lookup.items():
    golden = golden_lookup.get(family_id)
    if not golden or "error" in a2_result:
        continue

    expected = golden.get("expected_top_3", [])
    acceptable = golden.get("acceptable_rank_1", [])
    actual_top_3 = a2_result.get("top_3", [])

    expected_ids = [e["helper_id"] for e in expected]
    actual_ids = [a["helper_id"] for a in actual_top_3]

    # Rank 1 match — accept expected_rank_1 OR any id in acceptable_rank_1
    actual_rank1 = actual_ids[0] if actual_ids else None
    expected_rank1 = expected_ids[0] if expected_ids else None
    acceptable_rank1 = acceptable if acceptable else []

    rank1_match = (
        actual_rank1 is not None and
        (actual_rank1 == expected_rank1 or actual_rank1 in acceptable_rank1)
    )

    # Set match — same helpers regardless of order
    set_match = set(expected_ids) == set(actual_ids)

    # Overlap count — how many of the expected top 3 appear anywhere in actual top 3
    overlap = len(set(expected_ids) & set(actual_ids))

    # Exact order match
    exact_match = expected_ids == actual_ids

    match_results.append({
        "family_id": family_id,
        "difficulty": golden.get("difficulty"),
        "expected_ids": expected_ids,
        "acceptable_rank_1": acceptable_rank1,
        "actual_ids": actual_ids,
        "rank1_match": rank1_match,
        "set_match": set_match,
        "exact_match": exact_match,
        "overlap_count": overlap
    })

In [4]:
# Summary statistics
total = len(match_results)
rank1_hits = sum(1 for r in match_results if r["rank1_match"])
set_hits = sum(1 for r in match_results if r["set_match"])
exact_hits = sum(1 for r in match_results if r["exact_match"])
avg_overlap = sum(r["overlap_count"] for r in match_results) / total if total else 0

print("=== MATCH ACCURACY SUMMARY ===")
print(f"Total families evaluated:     {total}")
print(f"Rank 1 correct:               {rank1_hits}/{total} ({rank1_hits/total*100:.0f}%)")
print(f"Exact top-3 set match:        {set_hits}/{total} ({set_hits/total*100:.0f}%)")
print(f"Exact rank order match:       {exact_hits}/{total} ({exact_hits/total*100:.0f}%)")
print(f"Average helper overlap:       {avg_overlap:.1f}/3")

# Break down by difficulty
print("\n--- By Difficulty ---")
for difficulty in ["clear", "medium", "hard", "no_match"]:
    subset = [r for r in match_results if r["difficulty"] == difficulty]
    if not subset:
        continue
    r1 = sum(1 for r in subset if r["rank1_match"])
    ov = sum(r["overlap_count"] for r in subset) / len(subset)
    print(f"  {difficulty:<10} n={len(subset)}  rank1={r1}/{len(subset)}  avg_overlap={ov:.1f}")

# Save
with open(JUDGE_DIR / "match_accuracy.json", "w") as f:
    json.dump(match_results, f, indent=2)
print("\nSaved to outputs/judge/match_accuracy.json")

=== MATCH ACCURACY SUMMARY ===
Total families evaluated:     17
Rank 1 correct:               9/17 (53%)
Exact top-3 set match:        2/17 (12%)
Exact rank order match:       1/17 (6%)
Average helper overlap:       1.6/3

--- By Difficulty ---
  clear      n=7  rank1=3/7  avg_overlap=2.3
  medium     n=4  rank1=3/4  avg_overlap=1.2
  hard       n=3  rank1=2/3  avg_overlap=1.3
  no_match   n=3  rank1=1/3  avg_overlap=0.7

Saved to outputs/judge/match_accuracy.json


## 4. Agent 1 Confidence vs. Golden Set Expected Range
Check whether Agent 1's confidence scores fall within the golden set's expected confidence range.

In [5]:
print(f"{'Family':<8} {'Difficulty':<12} {'Expected Range':<20} {'Actual':>8} {'In Range':>10}")
print("-" * 65)

confidence_results = []
for family_id, care_profile in a1_lookup.items():
    golden = golden_lookup.get(family_id)
    if not golden or "error" in care_profile:
        continue

    expected_range = golden.get("expected_agent1_confidence", {})
    min_conf = expected_range.get("min", 0)
    max_conf = expected_range.get("max", 1)
    actual_conf = care_profile.get("confidence", None)
    in_range = actual_conf is not None and min_conf <= actual_conf <= max_conf

    confidence_results.append({
        "family_id": family_id,
        "difficulty": golden.get("difficulty"),
        "expected_min": min_conf,
        "expected_max": max_conf,
        "actual": actual_conf,
        "in_range": in_range
    })

    range_str = f"{min_conf:.2f} – {max_conf:.2f}"
    actual_str = f"{actual_conf:.2f}" if actual_conf is not None else "N/A"
    print(f"{family_id:<8} {golden.get('difficulty',''):<12} {range_str:<20} {actual_str:>8} {'✓' if in_range else '✗':>10}")

in_range_count = sum(1 for r in confidence_results if r["in_range"])
print(f"\nIn expected range: {in_range_count}/{len(confidence_results)}")

Family   Difficulty   Expected Range         Actual   In Range
-----------------------------------------------------------------
F001     clear        0.70 – 0.90              0.72          ✓
F002     clear        0.95 – 1.00              0.95          ✓
F003     clear        0.90 – 1.00              0.95          ✓
F004     clear        0.90 – 1.00              0.95          ✓
F005     clear        0.90 – 1.00              0.95          ✓
F006     medium       0.60 – 0.70              0.68          ✓
F007     medium       0.40 – 0.60              0.68          ✗
F008     medium       0.50 – 0.70              0.68          ✓
F009     hard         0.40 – 0.60              0.65          ✗
F010     medium       0.80 – 0.90              0.72          ✗
F011     no_match     0.80 – 1.00              0.92          ✓
F012     no_match     0.80 – 0.90              0.95          ✗
F013     no_match     0.80 – 1.00              0.92          ✓
F014     clear        0.90 – 1.00              0.92 

## 5. LLM-as-Judge — Agent 3 Explanation Quality
**Judge model:** GPT-4.1  
**Prompt version:** v1  
**Evaluates:** Specificity (1–5), Relevance (1–5), Factual Grounding (pass/fail), Empathy Tone (1–5)

In [6]:
JUDGE_PROMPT = """You are an independent evaluator assessing the quality of a care match explanation written for a family seeking elderly care support.

You will be given:
1. The family's care profile — their care needs, must-haves, deal-breakers, and how they described their situation
2. The helper's profile — their background, experience, skills, and certifications
3. The match explanation — the text written to help this family decide whether to reach out to this helper

Your job is to evaluate the explanation across 4 dimensions.

You MUST follow this process for every dimension:
  Step 1 — Quote or paraphrase the specific part of the explanation you are evaluating
  Step 2 — Reason about whether it meets the criterion
  Step 3 — Give your score or result

Do not give scores without completing Steps 1 and 2 first.

---

DIMENSION 1 — SPECIFICITY (1–5)
Does the explanation cite concrete, verifiable facts from the helper's profile, or does it use language that could apply to any helper?

  5 — Every meaningful claim cites a specific fact: years of experience, exact certification name, specific condition or prior role. No generic language present.
  4 — Mostly specific. One or two phrases are generic but do not mislead.
  3 — Mix of specific and generic. Some claims are grounded, others are floating.
  2 — Mostly generic. Specific facts used sparingly. Could describe most helpers.
  1 — Entirely generic. No facts from the helper's profile are cited.

---

DIMENSION 2 — RELEVANCE (1–5)
Does the explanation directly address what this specific family described — their care needs, their must-haves, and their stated concerns?

  5 — Explanation directly addresses the family's specific needs, mirrors their own language back to them, and speaks to their stated concerns and deal-breakers.
  4 — Addresses most of the family's key needs with minor gaps.
  3 — Addresses some needs but misses at least one significant concern the family raised.
  2 — Generic care match language. Does not engage with this family's specific situation.
  1 — Irrelevant or misaligned with what the family described.

---

DIMENSION 3 — FACTUAL GROUNDING (Pass / Fail)
Every claim in the explanation must be traceable to a specific fact in the helper's profile. Nothing invented or embellished.

  Pass — All claims can be verified against the helper's profile.
  Fail — One or more claims cannot be traced to the helper's profile. Any hallucinated or embellished fact = automatic Fail.

For this dimension, list every factual claim in the explanation and state whether each is traceable to the helper's profile.
Check years of experience, certification names, prior roles, skills, and any personality or behavioural claims.

---

DIMENSION 4 — EMPATHY TONE (1–5)
Is the tone appropriate for a family making a high-stakes care decision — warm and human, without being clinical, transactional, or overdramatic?

  5 — Warm, human, and holds the emotional weight of the decision without being dramatic or clinical. Reads like a thoughtful person wrote it.
  4 — Mostly appropriate tone. Minor lapses into clinical or transactional language.
  3 — Neutral — neither cold nor warm. Reads more like a professional summary than a human recommendation.
  2 — Clinical or transactional language dominates. Feels like a product listing.
  1 — Inappropriate tone — either cold and detached, or overdramatic and manipulative.

---

Return ONLY a JSON object in this format:

{
  "family_id": "...",
  "helper_id": "...",
  "evaluation": {
    "specificity": {
      "reasoning": "quote or paraphrase the relevant part of the explanation, then reason about whether claims are specific or generic",
      "score": 0
    },
    "relevance": {
      "reasoning": "quote or paraphrase the relevant part of the explanation, then reason about whether it addresses this family's specific needs and language",
      "score": 0
    },
    "factual_grounding": {
      "reasoning": "list each factual claim in the explanation and state whether it is traceable to the helper's profile",
      "result": "pass or fail",
      "failed_claims": []
    },
    "empathy_tone": {
      "reasoning": "quote or paraphrase the relevant part of the explanation, then reason about whether the tone is appropriate",
      "score": 0
    }
  },
  "overall_assessment": "2–3 sentences: what the explanation did well, and the single most important thing to improve"
}

Rules:
- Always complete reasoning before scoring. Never output a score without the reasoning that precedes it.
- Factual Grounding is binary. Partial hallucination is not a 2 or 3 — it is a Fail.
- failed_claims must always be returned — empty array if Pass, populated list if Fail.
- Do not reward an explanation for mentioning facts not relevant to this family's needs — relevance and specificity are separate dimensions.
- Do not penalise warm or emotional language if it is grounded in fact. Empathy and specificity are not in conflict."""


def run_judge(family_id: str, helper_id: str, care_profile: dict,
              helper_profile: dict, explanation: str) -> dict:
    """Run GPT-4.1 judge on one explanation. Returns evaluation JSON."""
    user_message = f"""Family care profile:
{json.dumps(care_profile, indent=2)}

Helper profile:
{json.dumps(helper_profile, indent=2)}

Match explanation to evaluate:
\"{explanation}\""""

    response = openai_client.chat.completions.create(
        model="gpt-4.1",
        messages=[
            {"role": "system", "content": JUDGE_PROMPT},
            {"role": "user", "content": user_message}
        ],
        response_format={"type": "json_object"},
        temperature=0
    )
    raw = response.choices[0].message.content
    try:
        parsed = json.loads(raw)
    except json.JSONDecodeError:
        parsed = {"error": "failed to parse JSON", "raw": raw}
    parsed["family_id"] = family_id
    parsed["helper_id"] = helper_id
    return parsed


print("Judge function defined.")

Judge function defined.


In [7]:
# Run Judge on all Agent 3 explanations
# Each family has 3 explanations — judge evaluates each one independently

judge_outputs = []

for family_id, a3_result in a3_lookup.items():
    care_profile = a1_lookup.get(family_id)
    recommendations = a3_result.get("recommendations", [])

    if not recommendations or not care_profile or "error" in care_profile:
        print(f"{family_id}: SKIPPED")
        continue

    for rec in recommendations:
        helper_id = rec.get("helper_id")
        explanation = rec.get("explanation", "")
        helper_profile = helper_lookup.get(helper_id)

        if not helper_profile or not explanation:
            continue

        print(f"Judging {family_id} / {helper_id} (rank {rec.get('rank')})...", end=" ")
        result = run_judge(family_id, helper_id, care_profile, helper_profile, explanation)
        result["rank"] = rec.get("rank")
        judge_outputs.append(result)

        eval_data = result.get("evaluation", {})
        s = eval_data.get("specificity", {}).get("score", "?")
        r = eval_data.get("relevance", {}).get("score", "?")
        fg = eval_data.get("factual_grounding", {}).get("result", "?")
        e = eval_data.get("empathy_tone", {}).get("score", "?")
        print(f"S:{s} R:{r} FG:{fg} E:{e}")
        time.sleep(1)

# Save
with open(JUDGE_DIR / "judge_outputs.json", "w") as f:
    json.dump(judge_outputs, f, indent=2)

print(f"\nJudge complete. {len(judge_outputs)} explanations evaluated.")
print(f"Saved to outputs/judge/judge_outputs.json")

Judging F001 / H004 (rank 1)... S:5 R:5 FG:Pass E:5
Judging F001 / H001 (rank 2)... S:5 R:5 FG:pass E:5
Judging F001 / H007 (rank 3)... S:5 R:5 FG:Pass E:5
Judging F002 / H003 (rank 1)... S:5 R:5 FG:pass E:5
Judging F002 / H002 (rank 2)... S:5 R:4 FG:pass E:5
Judging F002 / H009 (rank 3)... S:5 R:5 FG:pass E:5
Judging F003 / H004 (rank 1)... S:5 R:5 FG:Pass E:5
Judging F003 / H002 (rank 2)... S:5 R:5 FG:Pass E:5
Judging F003 / H009 (rank 3)... S:5 R:4 FG:pass E:5
Judging F004 / H002 (rank 1)... S:5 R:5 FG:Pass E:5
Judging F004 / H007 (rank 2)... S:5 R:4 FG:Pass E:5
Judging F004 / H009 (rank 3)... S:5 R:5 FG:pass E:5
Judging F005 / H004 (rank 1)... S:5 R:5 FG:pass E:5
Judging F005 / H001 (rank 2)... S:5 R:5 FG:pass E:5
Judging F005 / H003 (rank 3)... S:5 R:4 FG:pass E:5
Judging F006 / H007 (rank 1)... S:5 R:5 FG:pass E:5
Judging F006 / H013 (rank 2)... S:5 R:5 FG:pass E:5
Judging F006 / H002 (rank 3)... S:5 R:5 FG:pass E:5
Judging F007 / H003 (rank 1)... S:5 R:4 FG:Pass E:5
Judging F007

## 6. Score Analysis

In [8]:
# Aggregate judge scores across all explanations
valid = [j for j in judge_outputs if "evaluation" in j]

specificity_scores = [j["evaluation"]["specificity"]["score"] for j in valid]
relevance_scores = [j["evaluation"]["relevance"]["score"] for j in valid]
empathy_scores = [j["evaluation"]["empathy_tone"]["score"] for j in valid]
fg_results = [j["evaluation"]["factual_grounding"]["result"] for j in valid]

fg_pass = sum(1 for r in fg_results if r == "pass")
fg_fail = sum(1 for r in fg_results if r == "fail")

def avg(lst): return sum(lst) / len(lst) if lst else 0
def dist(lst):
    counts = {1:0, 2:0, 3:0, 4:0, 5:0}
    for s in lst: counts[s] = counts.get(s, 0) + 1
    return counts

print("=== JUDGE SCORE SUMMARY ===")
print(f"Explanations evaluated: {len(valid)}")
print()
print(f"Specificity    avg: {avg(specificity_scores):.2f}/5   distribution: {dist(specificity_scores)}")
print(f"Relevance      avg: {avg(relevance_scores):.2f}/5   distribution: {dist(relevance_scores)}")
print(f"Empathy Tone   avg: {avg(empathy_scores):.2f}/5   distribution: {dist(empathy_scores)}")
print(f"Factual Ground pass: {fg_pass}/{len(valid)}  fail: {fg_fail}/{len(valid)}")

=== JUDGE SCORE SUMMARY ===
Explanations evaluated: 51

Specificity    avg: 4.94/5   distribution: {1: 0, 2: 0, 3: 0, 4: 3, 5: 48}
Relevance      avg: 4.59/5   distribution: {1: 0, 2: 0, 3: 1, 4: 19, 5: 31}
Empathy Tone   avg: 5.00/5   distribution: {1: 0, 2: 0, 3: 0, 4: 0, 5: 51}
Factual Ground pass: 36/51  fail: 1/51


In [9]:
# Failures — list all factual grounding failures and low scores
print("=== FACTUAL GROUNDING FAILURES ===")
failures = [j for j in valid if j["evaluation"]["factual_grounding"]["result"] == "fail"]
if not failures:
    print("None — all explanations passed factual grounding.")
else:
    for f in failures:
        print(f"\n{f['family_id']} / {f['helper_id']} (rank {f.get('rank')})")
        failed_claims = f["evaluation"]["factual_grounding"].get("failed_claims", [])
        for claim in failed_claims:
            print(f"  - {claim}")

print()
print("=== LOW RELEVANCE SCORES (≤ 2) ===")
low_relevance = [j for j in valid if j["evaluation"]["relevance"]["score"] <= 2]
if not low_relevance:
    print("None.")
else:
    for j in low_relevance:
        score = j["evaluation"]["relevance"]["score"]
        reasoning = j["evaluation"]["relevance"]["reasoning"]
        print(f"\n{j['family_id']} / {j['helper_id']} — score: {score}")
        print(f"  {reasoning[:200]}...")

print()
print("=== LOW SPECIFICITY SCORES (≤ 2) ===")
low_specificity = [j for j in valid if j["evaluation"]["specificity"]["score"] <= 2]
if not low_specificity:
    print("None.")
else:
    for j in low_specificity:
        score = j["evaluation"]["specificity"]["score"]
        print(f"{j['family_id']} / {j['helper_id']} — score: {score}")

=== FACTUAL GROUNDING FAILURES ===

F016 / H004 (rank 2)
  - Families who have gone through the experience of finding the right helper often say they wished they'd found someone like Rosa sooner.

=== LOW RELEVANCE SCORES (≤ 2) ===
None.

=== LOW SPECIFICITY SCORES (≤ 2) ===
None.


In [10]:
# Score breakdown by difficulty level
print("=== SCORES BY DIFFICULTY ===")

for difficulty in ["clear", "medium", "hard", "no_match"]:
    family_ids = [
        f["family_id"] for f in families
        if f.get("difficulty") == difficulty
    ]
    subset = [j for j in valid if j["family_id"] in family_ids]
    if not subset:
        continue

    s_scores = [j["evaluation"]["specificity"]["score"] for j in subset]
    r_scores = [j["evaluation"]["relevance"]["score"] for j in subset]
    e_scores = [j["evaluation"]["empathy_tone"]["score"] for j in subset]
    fg_p = sum(1 for j in subset if j["evaluation"]["factual_grounding"]["result"] == "pass")

    print(f"\n  {difficulty.upper()} (n={len(subset)} explanations)")
    print(f"    Specificity:    {avg(s_scores):.2f}")
    print(f"    Relevance:      {avg(r_scores):.2f}")
    print(f"    Empathy Tone:   {avg(e_scores):.2f}")
    print(f"    Factual Ground: {fg_p}/{len(subset)} pass")

=== SCORES BY DIFFICULTY ===

  CLEAR (n=21 explanations)
    Specificity:    4.90
    Relevance:      4.67
    Empathy Tone:   5.00
    Factual Ground: 14/21 pass

  MEDIUM (n=12 explanations)
    Specificity:    4.92
    Relevance:      4.50
    Empathy Tone:   5.00
    Factual Ground: 8/12 pass

  HARD (n=9 explanations)
    Specificity:    5.00
    Relevance:      4.44
    Empathy Tone:   5.00
    Factual Ground: 8/9 pass

  NO_MATCH (n=9 explanations)
    Specificity:    5.00
    Relevance:      4.67
    Empathy Tone:   5.00
    Factual Ground: 6/9 pass


## 7. Failure Taxonomy
Map observed failures to the taxonomy labels defined in the prompt log.

In [11]:
# Failure taxonomy labels:
# A — Hallucination     (factual_grounding = fail)
# B — Generic           (specificity <= 2)
# C — Relevance miss    (relevance <= 2)
# D — Tone mismatch     (empathy_tone <= 2)
# E — Wrong rank        (rank 1 in Agent 2 != golden rank 1)

taxonomy = {"A": [], "B": [], "C": [], "D": [], "E": []}

for j in valid:
    fid = j["family_id"]
    hid = j["helper_id"]
    label = f"{fid}/{hid}"

    if j["evaluation"]["factual_grounding"]["result"] == "fail":
        taxonomy["A"].append(label)
    if j["evaluation"]["specificity"]["score"] <= 2:
        taxonomy["B"].append(label)
    if j["evaluation"]["relevance"]["score"] <= 2:
        taxonomy["C"].append(label)
    if j["evaluation"]["empathy_tone"]["score"] <= 2:
        taxonomy["D"].append(label)

# Wrong rank — E
for r in match_results:
    if not r["rank1_match"]:
        taxonomy["E"].append(r["family_id"])

print("=== FAILURE TAXONOMY ===")
labels = {
    "A": "Hallucination",
    "B": "Generic language",
    "C": "Relevance miss",
    "D": "Tone mismatch",
    "E": "Wrong rank 1"
}
for code, name in labels.items():
    instances = taxonomy[code]
    print(f"  {code} — {name}: {len(instances)} instance(s)")
    for i in instances:
        print(f"      {i}")

# Save taxonomy
with open(JUDGE_DIR / "failure_taxonomy.json", "w") as f:
    json.dump(taxonomy, f, indent=2)
print("\nSaved to outputs/judge/failure_taxonomy.json")

=== FAILURE TAXONOMY ===
  A — Hallucination: 1 instance(s)
      F016/H004
  B — Generic language: 0 instance(s)
  C — Relevance miss: 0 instance(s)
  D — Tone mismatch: 0 instance(s)
  E — Wrong rank 1: 10 instance(s)
      F001
      F002
      F005
      F007
      F009
      F011
      F013
      F015
      F016
      F017

Saved to outputs/judge/failure_taxonomy.json


## 8. Overall Eval Summary
Consolidated view to populate the Iteration Summary Table in prompt_log.md.

In [12]:
print("=== OVERALL EVAL SUMMARY ===")
print("Copy these scores into the Iteration Summary Table in evals/prompts/prompt_log.md")
print()
print(f"  Agent 1 confidence in range:  {in_range_count}/{len(confidence_results)}")
print(f"  Agent 2 rank-1 match:         {rank1_hits}/{total} ({rank1_hits/total*100:.0f}%)")
print(f"  Agent 2 avg helper overlap:   {avg_overlap:.1f}/3")
print()
print(f"  Agent 3 — Specificity:        {avg(specificity_scores):.2f}/5")
print(f"  Agent 3 — Relevance:          {avg(relevance_scores):.2f}/5")
print(f"  Agent 3 — Factual Grounding:  {fg_pass}/{len(valid)} pass")
print(f"  Agent 3 — Empathy Tone:       {avg(empathy_scores):.2f}/5")
print()
print(f"  Most common failure:          ", end="")
most_common = max(taxonomy, key=lambda k: len(taxonomy[k]))
print(f"{most_common} — {labels[most_common]} ({len(taxonomy[most_common])} instances)")

=== OVERALL EVAL SUMMARY ===
Copy these scores into the Iteration Summary Table in evals/prompts/prompt_log.md

  Agent 1 confidence in range:  13/17
  Agent 2 rank-1 match:         7/17 (41%)
  Agent 2 avg helper overlap:   1.7/3

  Agent 3 — Specificity:        4.94/5
  Agent 3 — Relevance:          4.59/5
  Agent 3 — Factual Grounding:  36/51 pass
  Agent 3 — Empathy Tone:       5.00/5

  Most common failure:          E — Wrong rank 1 (10 instances)
